In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import OrdinalEncoder , StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score , classification_report , precision_recall_curve, confusion_matrix , roc_curve , auc

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier , VotingClassifier ,  AdaBoostClassifier , GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV

In [ ]:
data = pd.read_csv('Loan Status.csv')

## Data Analysis and Feature Engineering
 ---

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data.dropna(inplace = True)
data = data.drop(columns=['Loan_ID'])

In [ ]:
enc = OrdinalEncoder() # As all are just categories
scaler = StandardScaler() 
numeric_columns = list(data.select_dtypes(['int64' , 'float64']).columns)
categorical_columns = list(data.select_dtypes('object').columns)

In [ ]:
corr_matrix = data[numeric_columns].corr()
sns.heatmap(corr_matrix , annot = True , fmt = '.2f') 
# Couldn't draw much information from the Correlation matrix

In [ ]:
data_pipeline = ColumnTransformer([
    ('Normalization' , scaler , numeric_columns),
    ('OrdinalEncoder' , enc , categorical_columns)
])
scaled_data = data_pipeline.fit_transform(data)

In [ ]:
preprocessed_data = pd.DataFrame(scaled_data , columns = numeric_columns + categorical_columns)

In [ ]:
target = preprocessed_data['Loan_Status']
preprocessed_data.drop(columns = ['Loan_Status'] , inplace = True)

In [ ]:
x_train , x_test , y_train , y_test = train_test_split(np.array(preprocessed_data) , np.array(target) , test_size=0.2)

## Training Different Models
---

In [ ]:
# Initializing different Models
voting_clf = VotingClassifier(estimators = [('model1' , SVC()), ('model2' , LogisticRegression()) , ('model3', RandomForestClassifier()) , ('model4' , DecisionTreeClassifier())])
MODEL_SET ={
                'log_clf' : LogisticRegression() , 
                'xg_boost' : XGBClassifier() , 
                'svc' : SVC(),
                'd-trees' : DecisionTreeClassifier() , 
                'RF_clf' : RandomForestClassifier() , 
                'GB_clf' : GradientBoostingClassifier(),
                'Ada_clf': AdaBoostClassifier(),
                'voting_clf' : voting_clf
            }

In [ ]:
# training different models
best_model_names = []
for name , model in MODEL_SET.items():
    model.fit(x_train, y_train)
    train_predict = model.predict(x_train)
    acc = accuracy_score(train_predict , y_train)
    if acc > 0.95:
        best_model_names.append(name)
    print(f'the training error of {name} is : {accuracy_score(train_predict , y_train)}')

### Analysis
   <p> Based on the performance of the models without optimization
   We can consider 5 models that gave good results
   XGboost , DecisionTrees , RandomForest , Voting Classifier<p>

## Hyperparameter tuning
---

In [ ]:
algos = {
    'decision_tree': {
        'model': DecisionTreeClassifier(),
        'params': {
            'criterion': ['gini', 'entropy', 'log_loss'],
            'splitter': ['best', 'random']
        }
    },
    'RandomForestClassifier': {
        'model': RandomForestClassifier(),
        'params': {
            'n_estimators': [10, 50, 100, 130],
            'max_depth': range(2, 5, 1),
            'criterion': ['gini', 'entropy', 'log_loss'],
            'max_features': ['sqrt', 'log2']
        }
    },
    'XGBClassifier': {
        'model': XGBClassifier(),
        'params': {
            'learning_rate': [0.5, 0.1, 0.01, 0.001],
            'max_depth': [2, 3],
            'n_estimators': [10, 50, 100, 200]
        }
    },
    'VotingClassifier': {
        'model': VotingClassifier([
            ('model1', SVC(probability=True)),
            ('model2', LogisticRegression()),
            ('model3', RandomForestClassifier()),
            ('model4', DecisionTreeClassifier())
        ]),
        'params': {
            'voting': ['soft']
        }
    }
}

final_models = []

for algorithm_name, config in algos.items():
    model = config['model']
    param_grid = config['params']
    
    # Perform grid search
    gscv = GridSearchCV(model, param_grid, cv=10)
    gscv.fit(x_train, y_train)
    
    final_models.append({
        'name': algorithm_name,
        'model': gscv.best_estimator_,
        'score': gscv.best_score_
    })

In [ ]:
for item in final_models:
    print(f"{item['name']} best_score : {item['score']}")
    # Lets evaluate over test data
    prediction = item['model'].predict(x_test)
    print(f'Test Accuracy : {accuracy_score(prediction , y_test)}\n') 

In [ ]:
# Classification Report For Random Forest Regressor
model = final_models[2]['model']
prediction = model.predict(x_test)
print(f'XGBoost Classifier: \n {classification_report(prediction , y_test)}')


## Visulizations
---

In [ ]:
confusion_mat = confusion_matrix(prediction , y_test)
sns.heatmap(confusion_mat , annot = True)
plt.title('Confusion Matrix')

In [ ]:
# Lets draw precision , Recall curve for the Model
proba = model.predict_proba(x_test)
proba = np.max(proba , axis = 1)

precisions , recalls , thresholds = precision_recall_curve(y_test , proba)
plt.plot(recalls, precisions, lw=1, color='blue')
plt.fill_between(recalls, precisions, -1, facecolor='lightblue', alpha=0.5)
plt.title('PR Curve for Random Forests classifier')
plt.xlabel('Recall (TPR)')
plt.ylabel('Precision')
plt.show()

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, proba)

auc_value = auc(fpr , tpr)
plt.plot(fpr, tpr , color='green' , label = f'AUC : {auc_value:.2f}')
plt.plot(fpr , fpr , '--' , color = 'black' , label = "Normal Classifier")
plt.title('ROC Curve for RF classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.legend()
plt.show()